# **예측 성능을 수요 계획 판단으로 연결**

모델 점수가 좋아졌다는 사실만으로 발주량을 정할 수는 없습니다. 이 노트북은 저장된 예측을 같은 날짜·매장·상품군에서 비교하고, **어떤 수요에 LightGBM이 도움이 되는지** 확인합니다.

7일 합계, 수요 규모, 매장·상품군, 군집별 오류를 본 뒤 검토할 사례를 고릅니다. 실제 판매량은 회고적 비교에만 쓰며 재고·공급 자료가 없으므로 최적 발주량이나 비용 절감을 주장하지 않습니다.

| 결과에서 볼 값 | 업무에서의 의미 |
|---|---|
| 7일 WAPE | 실제 7일 판매량에 비해 예측이 얼마나 빗나갔는지 나타내는 비율. 낮을수록 좋음 |
| Bias | 예측 합계가 실제보다 큰지 작은지. 양수면 과대예측 |
| 판매량 비중 | 해당 매장·상품군 집단이 전체 수요에서 차지하는 몫 |

> [그림 추가 자리: 예측 → 기준선 비교 → 검토 대상 선정 → 담당자 확인의 의사결정 흐름]

## **환경 준비**

별도의 00 노트북에서 저장소를 My Drive에 clone·갱신한 뒤 실행합니다. Drive를 연결하고 `/content/drive/MyDrive/retail-demand-forecasting`을 작업 폴더로 사용합니다.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive/")
except ImportError:
    pass

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

### **시각화 패키지**

모델을 설치하거나 학습하지 않습니다. 결과 Parquet과 한글 그래프를 읽고 그리는 패키지만 준비합니다.

Colab에서는 **환경 준비 → 패키지 설치 → Library** 셀 순서로 실행합니다. 런타임을 재시작했다면 설치 셀도 다시 실행해야 `retail_forecast`를 가져올 수 있습니다.

In [ ]:
%pip install -q seaborn koreanize-matplotlib pyarrow

%pip install -q -e "{REPO}" --no-deps

## **Library**


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from importlib.util import find_spec
from matplotlib import font_manager

from retail_forecast.comparison import compare_on_common_rows, load_common_predictions
from retail_forecast.metrics import regression_metrics
from retail_forecast.neural_data import KEYS, summarize_neural_predictions

## **Utils**

공통 행 비교, 기준선 추가, 그룹별 오류 계산에 필요한 짧은 함수만 둡니다.


In [ ]:
# 저장된 모델별 Valid/Test 결과 경로를 모읍니다.
def prediction_paths(ml_dir, dl_dir, phase):
    return {
        "LightGBM": ml_dir / f"{phase}_predictions_lightgbm.csv",
        "CatBoost": ml_dir / f"{phase}_predictions_catboost.csv",
        "NHITS": dl_dir / "nhits" / f"{phase}_predictions_nhits.parquet",
        "TCN": dl_dir / "tcn" / f"{phase}_predictions_tcn.parquet",
        "TFT": dl_dir / "tft" / f"{phase}_predictions_tft.parquet",
    }

In [ ]:
# 모델 예측과 두 단순 기준선을 같은 행에서 비교합니다.
def compare_with_baselines(paths):
    model_metrics = compare_on_common_rows(paths)
    reference = pd.read_csv(paths["LightGBM"], parse_dates=["date", "target_date"])
    baselines = {
        "전주 동일 요일": "sales_same_weekday_last_week",
        "최근 4주 동일 요일 평균": "sales_same_weekday_4w_mean",
    }
    baseline_metrics = []
    for name, column in baselines.items():
        frame = reference[KEYS + ["target_sales", column]].rename(columns={column: "prediction"})
        metrics, _ = summarize_neural_predictions(frame, name)
        baseline_metrics.append(metrics)
    return pd.concat([model_metrics, *baseline_metrics], ignore_index=True)

In [ ]:
# 같은 그룹의 예측 오류와 판매량을 집계합니다.
def group_scores(frame, groups):
    errors = frame.assign(
        model_absolute=(frame["prediction"] - frame["target_sales"]).abs(),
        baseline_absolute=(frame["sales_same_weekday_4w_mean"] - frame["target_sales"]).abs(),
        model_signed=frame["prediction"] - frame["target_sales"],
    )
    result = errors.groupby(groups, observed=True).agg(
        rows=("target_sales", "size"),
        actual_sales=("target_sales", "sum"),
        model_absolute=("model_absolute", "sum"),
        baseline_absolute=("baseline_absolute", "sum"),
        model_signed=("model_signed", "sum"),
    ).reset_index()
    denominator = result["actual_sales"].replace(0, np.nan)
    result["lightgbm_wape"] = 100 * result["model_absolute"] / denominator
    result["baseline_wape"] = 100 * result["baseline_absolute"] / denominator
    result["gain_pp"] = result["baseline_wape"] - result["lightgbm_wape"]
    result["excess_absolute_error"] = result["model_absolute"] - result["baseline_absolute"]
    result["actual_share_pct"] = 100 * result["actual_sales"] / result["actual_sales"].sum()
    result["lightgbm_bias"] = 100 * result["model_signed"] / denominator
    return result.drop(columns=["model_absolute", "baseline_absolute", "model_signed"])

In [ ]:
# 한 기준일의 1~7일 예측을 7일 합계로 묶습니다.
def weekly_predictions(frame):
    keys = ["date", "store_nbr", "family"]
    assert frame.groupby(keys, observed=True)["forecast_horizon"].nunique().eq(7).all()
    return frame.groupby(keys, observed=True)[
        ["target_sales", "prediction", "sales_same_weekday_4w_mean"]
    ].sum(min_count=7).reset_index()

In [ ]:
# Valid에서 정한 경계로 예상 수요 규모를 구분합니다.
def assign_demand_tier(frame, cutoffs):
    result = frame.copy()
    demand = result["sales_same_weekday_4w_mean"]
    result["demand_tier"] = pd.Categorical(
        np.select([demand.le(cutoffs[0]), demand.le(cutoffs[1])],
                  ["낮음", "중간"], default="높음"),
        categories=["낮음", "중간", "높음"], ordered=True,
    )
    return result

In [ ]:
# 모델마다 같은 기준선과 7일 합계 행을 만듭니다.
def model_weekly_predictions(paths, reference, cutoffs):
    baseline = reference[KEYS + ["sales_same_weekday_4w_mean"]]
    results = {}
    for name, path in paths.items():
        predictions = load_common_predictions(path).merge(
            baseline, on=KEYS, validate="one_to_one",
        )
        assert len(predictions) == len(reference), f"{name}: 평가 행이 다릅니다."
        results[name] = assign_demand_tier(weekly_predictions(predictions), cutoffs)
    return results

In [ ]:
# 각 모델을 같은 그룹과 지표로 비교합니다.
def model_group_scores(weekly_by_model, groups):
    tables = []
    for name, weekly in weekly_by_model.items():
        table = group_scores(weekly, groups).rename(columns={
            "lightgbm_wape": "model_wape", "lightgbm_bias": "model_bias",
        })
        table.insert(0, "model", name)
        tables.append(table)
    return pd.concat(tables, ignore_index=True)

In [ ]:
# Valid에서 선택한 낮은 수요 대체 규칙을 Test에 적용합니다.
def model_fallback_scores(valid_by_model, test_by_model):
    rows = []
    for name, valid in valid_by_model.items():
        valid_low = valid.loc[valid["demand_tier"].eq("낮음")]
        model_error = (valid_low["prediction"] - valid_low["target_sales"]).abs().sum()
        baseline_error = (
            valid_low["sales_same_weekday_4w_mean"] - valid_low["target_sales"]
        ).abs().sum()
        use_fallback = bool(model_error > baseline_error)
        for phase, frame in [("Valid", valid), ("Test", test_by_model[name])]:
            fallback = frame["demand_tier"].eq("낮음") & use_fallback
            forecast = frame["prediction"].where(
                ~fallback, frame["sales_same_weekday_4w_mean"],
            )
            for method, prediction in [("모델", frame["prediction"]),
                                       ("4주 기준선", frame["sales_same_weekday_4w_mean"]),
                                       ("낮은 수요 대체", forecast)]:
                rows.append({
                    "phase": phase, "model": name, "method": method,
                    "rows": len(frame), "low_demand_fallback": use_fallback,
                    **regression_metrics(frame["target_sales"], prediction),
                })
    return pd.DataFrame(rows)

## **결과 경로**

05와 07에서 사용한 실행 이름을 그대로 적습니다. 필요한 예측 파일이 없으면 해당 실행의 **결과 파일**을 먼저 준비하세요. 이 노트북은 학습을 대신 실행하지 않습니다.


In [ ]:
font_path = Path(find_spec("koreanize_matplotlib").origin).parent / "fonts" / "NanumGothic.ttf"
font_manager.fontManager.addfont(font_path)
sns.set_theme(style="whitegrid", font=font_manager.FontProperties(fname=font_path).get_name())
plt.rcParams["axes.unicode_minus"] = False

ML_RUN_NAME = "full_history_time_split_fixed_i600"
DL_RUN_NAME = "neural_time_split_val_loss"

ml_dir = REPO / "outputs" / ML_RUN_NAME
dl_dir = REPO / "outputs" / DL_RUN_NAME
analysis_dir = REPO / "outputs" / "decision_analysis" / f"{ML_RUN_NAME}__{DL_RUN_NAME}"
analysis_dir.mkdir(parents=True, exist_ok=True)

valid_paths = prediction_paths(ml_dir, dl_dir, "validation")
test_paths = prediction_paths(ml_dir, dl_dir, "test")
missing = [path for path in [*valid_paths.values(), *test_paths.values()] if not path.is_file()]
assert not missing, f"필요한 결과 파일이 없습니다: {missing}"

## **1. 같은 수요에서 모델과 기준선 비교**

- 다섯 모델과 두 기준선을 **같은 날짜·매장·상품군·horizon**에서 비교합니다. 표의 `daily`와 `7_day_sum`은 서로 다른 평가 단위입니다.
- 7일 합계 WAPE는 다음 주 수요 계획의 출발점입니다. Valid는 설정 검토, Test는 고정한 설정의 성능 확인에 사용합니다.

In [ ]:
valid_table = compare_with_baselines(valid_paths)
test_table = compare_with_baselines(test_paths)

valid_table.to_csv(analysis_dir / "comparison_with_baselines_valid.csv", index=False)
test_table.to_csv(analysis_dir / "comparison_with_baselines_test.csv", index=False)

overall = pd.concat([
    valid_table.assign(phase="Valid"),
    test_table.assign(phase="Test"),
], ignore_index=True)
overall = overall.loc[
    overall["forecast_horizon"].eq(0),
    ["phase", "model", "scope", "rows", "mae", "rmse", "wape", "bias"],
].sort_values(["phase", "scope", "wape"])
overall

### **Test 성능을 한눈에 비교**

- 막대가 짧을수록 WAPE가 낮습니다. 실제 수요량 대비 전체 절대오차를 비교하므로, **같은 Test 행에서의 기준선 개선**에 주목합니다.
- 좋은 점수만으로 발주량을 정하지 않고, 이후 상품군·매장·수요 유형별 Bias와 실패 사례를 확인합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(
    data=overall.loc[overall["phase"].eq("Test")],
    x="wape", y="model", hue="scope", ax=ax,
)
ax.set(xlabel="WAPE (%)", ylabel="", title="동일 Test 행: 모델과 주간 기준선")
fig.tight_layout()
fig.savefig(analysis_dir / "test_wape_comparison.png", dpi=160)
plt.show()


## **2. LightGBM이 기준선보다 나쁜 구간 찾기**

- `gain_pp = 기준선 WAPE − LightGBM WAPE`입니다. 음수이면 같은 Test 구간에서 LightGBM이 4주 기준선보다 나쁩니다.
- `actual_share_pct`는 그 구간이 차지하는 **실제 판매 수량 비중**입니다. 비중이 극히 작으면 WAPE가 매우 커도 전체 계획에 미치는 영향은 작을 수 있습니다.
- 이 구간 분석은 Test 결과의 **진단**이며, 이를 보고 새 대체 규칙을 고르면 별도 미래 구간에서 다시 검증해야 합니다.

| 열 | 의미 | 읽을 때 주의할 점 |
|---|---|---|
| `actual_share_pct` | Test 전체 판매 수량 중 해당 그룹의 비중 | 매출·이익 비중은 아님 |
| `lightgbm_wape`, `baseline_wape` | 각 그룹의 실제 판매 수량 대비 절대오차 비율 | 실제 수량이 거의 0이면 불안정 |
| `gain_pp` | 기준선 WAPE − 모델 WAPE | 양수면 모델 개선, 음수면 악화 |
| `excess_absolute_error` | 모델 절대오차 합 − 기준선 절대오차 합 | 양수면 모델 때문에 늘어난 오차 수량 |
| `lightgbm_bias` | `(예측 합 − 실제 합) / 실제 합 × 100` | 양수는 과대예측 |


### **예측 거리별 개선**

- 1~7일 중 멀리 내다보는 날짜에서 기준선 대비 개선이 줄어드는지 확인합니다.


In [ ]:
test_predictions = pd.read_csv(test_paths["LightGBM"], parse_dates=["date", "target_date"])

horizon_scores = group_scores(test_predictions, ["forecast_horizon"])
family_scores = group_scores(test_predictions, ["family"])
store_scores = group_scores(test_predictions, ["store_nbr"])

for name, table in {
    "horizon": horizon_scores,
    "family": family_scores,
    "store": store_scores,
}.items():
    table.to_csv(analysis_dir / f"test_error_by_{name}.csv", index=False)

horizon_scores

### **상품군별 취약 구간**

- `gain_pp`가 낮은 상품군을 먼저 보여줍니다. 판매량 비중이 작은 영역은 이후 7일 합계 표와 함께 읽습니다.


In [ ]:
family_error_view = (family_scores.sort_values("gain_pp").head(12))
family_error_view


### **매장별 취약 구간**

- 점포별 WAPE 격차를 확인합니다. 이 표만으로 공급량을 조정하지는 않습니다.


In [ ]:
store_error_view = (store_scores.sort_values("gain_pp").head(12))
store_error_view


In [ ]:
# 기준선 대비 개선폭과 예측 방향을 같은 horizon에서 읽습니다.
# 양의 Bias는 과대예측이며, 개선폭이 음수면 기준선보다 나쁩니다.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=horizon_scores, x="forecast_horizon", y="gain_pp", marker="o", ax=axes[0])
sns.lineplot(data=horizon_scores, x="forecast_horizon", y="lightgbm_bias", marker="o", ax=axes[1])
axes[0].axhline(0, color="gray", linewidth=1)
axes[0].set(xlabel="예측 거리 (일)", ylabel="WAPE 개선 (pp)", title="4주 기준선 대비 개선")
axes[1].set(xlabel="예측 거리 (일)", ylabel="Bias (%)", title="LightGBM 과대·과소예측")
fig.tight_layout()
fig.savefig(analysis_dir / "test_horizon_errors.png", dpi=160)
plt.show()


## **3. 수요 규모별 7일 합계 오류**

- 수요 규모의 경계는 **Valid의 4주 기준선 예상 수요**로 정하고 Test에 그대로 적용합니다. 실제 Test 판매량으로 그룹을 나누지 않습니다.
- 7일 합계의 판매량 비중·WAPE·추가 절대오차를 함께 보고, 어떤 규모에서 모델이 계획에 도움 되는지 판단합니다.

### **Valid의 수요 규모별 오류**

- 수요 규모 경계와 대체 규칙을 고를 때 참조하는 표입니다. 이 결과로 Test 설정을 다시 고르지 않습니다.


In [ ]:
valid_predictions = pd.read_csv(valid_paths["LightGBM"], parse_dates=["date", "target_date"])
valid_weekly = weekly_predictions(valid_predictions)
weekly = weekly_predictions(test_predictions)
cutoffs = valid_weekly["sales_same_weekday_4w_mean"].quantile([1 / 3, 2 / 3]).to_numpy()
valid_weekly = assign_demand_tier(valid_weekly, cutoffs)
weekly = assign_demand_tier(weekly, cutoffs)

valid_tier_scores = group_scores(valid_weekly, ["demand_tier"])
tier_scores = group_scores(weekly, ["demand_tier"])
weekly_family_scores = group_scores(weekly, ["family"])
weekly_store_scores = group_scores(weekly, ["store_nbr"])

for name, table in {
    "valid_7day_error_by_demand_tier": valid_tier_scores,
    "test_7day_error_by_demand_tier": tier_scores,
    "test_7day_error_by_family": weekly_family_scores,
    "test_7day_error_by_store": weekly_store_scores,
}.items():
    table.to_csv(analysis_dir / f"{name}.csv", index=False)

valid_tier_scores

### **Test의 수요 규모별 오류**

- Valid에서 정한 같은 경계를 적용해 7일 합계의 오류·판매량 비중을 확인합니다.


In [ ]:
tier_scores

### **7일 합계: 상품군별 추가 오차**

- `excess_absolute_error`가 큰 상품군부터 검토합니다. 양수는 4주 기준선보다 모델의 절대오차가 더 컸다는 뜻입니다.


In [ ]:
weekly_family_view = (weekly_family_scores.sort_values("excess_absolute_error", ascending=False).head(12))
weekly_family_view


### **7일 합계: 매장별 추가 오차**

- 같은 추가 오차를 점포별로 보아 집중 검토할 곳을 찾습니다.


In [ ]:
weekly_store_view = (weekly_store_scores.sort_values("excess_absolute_error", ascending=False).head(12))
weekly_store_view


## **4. 낮은 수요·취약 구간 진단**

낮은 수요 구간에서 LightGBM이 기준선보다 얼마나 손해를 보는지 **추가 절대오차** 순서로 확인합니다. WAPE가 매우 큰 상품군이라도 실제 판매량 비중과 추가 절대오차가 작으면 전체 운영 영향은 제한적입니다. 이는 원인 탐색표이며, Test를 보고 제외 상품군을 확정하지 않습니다.

### **낮은 수요의 상품군별 오류**

- 낮은 예상 수요에서 기준선보다 나빠진 상품군을 찾습니다. 0에 가까운 실제 수요에서는 WAPE보다 추가 오차 수량을 먼저 봅니다.


In [ ]:
low_weekly = weekly.loc[weekly["demand_tier"].eq("낮음")]
low_family = group_scores(low_weekly, ["family"])
low_store = group_scores(low_weekly, ["store_nbr"])

low_family.to_csv(analysis_dir / "test_low_demand_by_family.csv", index=False)
low_store.to_csv(analysis_dir / "test_low_demand_by_store.csv", index=False)
low_family_view = (low_family.sort_values("excess_absolute_error", ascending=False).head(12))
low_family_view


### **낮은 수요의 매장별 오류**

- 낮은 수요의 점포별 실패가 소수 점포에 집중되는지 확인합니다.


In [ ]:
low_store_view = (low_store.sort_values("excess_absolute_error", ascending=False).head(12))
low_store_view


## **5. 기준선 대체 규칙 시험**

Valid의 **낮은 수요 구간**에서 기준선이 더 정확했을 때만 해당 구간을 4주 기준선으로 대체합니다. 경계와 선택 여부를 Valid에서 정한 뒤 Test에 그대로 적용합니다. Test는 이미 여러 차례 확인했으므로 아래 Test 점수는 독립적인 성능 확증이 아니라 **탐색적 점검**입니다. 실제 적용 여부는 다른 과거 시점에서 다시 검증해야 합니다.

In [ ]:
valid_low = valid_tier_scores.loc[valid_tier_scores["demand_tier"].eq("낮음")].iloc[0]
use_low_fallback = bool(valid_low["excess_absolute_error"] > 0)

rule_rows = []
for phase, frame in [("Valid", valid_weekly), ("Test", weekly)]:
    fallback = frame["demand_tier"].eq("낮음") & use_low_fallback
    candidates = {
        "LightGBM": frame["prediction"],
        "4주 기준선": frame["sales_same_weekday_4w_mean"],
        "Valid 선택 규칙": frame["prediction"].where(
            ~fallback, frame["sales_same_weekday_4w_mean"]
        ),
    }
    for name, forecast in candidates.items():
        rule_rows.append({
            "phase": phase, "method": name, "rows": len(frame),
            "low_demand_fallback": use_low_fallback,
            **regression_metrics(frame["target_sales"], forecast),
        })

rule_scores = pd.DataFrame(rule_rows)
rule_scores.to_csv(analysis_dir / "weekly_fallback_exploration.csv", index=False)
rule_scores

## **6. 다섯 모델의 같은 구간 비교**

LightGBM·CatBoost·N-HiTS·TCN·TFT의 저장된 Valid/Test 예측에 **같은 4주 기준선과 Valid에서 정한 수요 경계**를 붙입니다. 7일 합계의 수요 규모·상품군·매장별 WAPE, Bias, 기준선 대비 추가 절대오차를 비교합니다. `gain_pp`가 음수이면 해당 모델이 기준선보다 나쁩니다. 이 셀은 모델을 다시 학습하지 않습니다.

### **다섯 모델의 수요 규모별 비교**

- 동일한 Valid 경계·Test 행에서 모델별 7일 합계 성능을 비교합니다. 모델마다 평가 대상을 바꾸지 않습니다.


In [ ]:
valid_model_weekly = model_weekly_predictions(valid_paths, valid_predictions, cutoffs)
test_model_weekly = model_weekly_predictions(test_paths, test_predictions, cutoffs)

all_tier_scores = model_group_scores(test_model_weekly, ["demand_tier"])
all_family_scores = model_group_scores(test_model_weekly, ["family"])
all_store_scores = model_group_scores(test_model_weekly, ["store_nbr"])

for name, table in {
    "all_models_test_7day_by_demand_tier": all_tier_scores,
    "all_models_test_7day_by_family": all_family_scores,
    "all_models_test_7day_by_store": all_store_scores,
}.items():
    table.to_csv(analysis_dir / f"{name}.csv", index=False)

model_tier_view = (all_tier_scores[[
    "model", "demand_tier", "rows", "actual_share_pct", "model_wape",
    "baseline_wape", "gain_pp", "model_bias",
]])
model_tier_view


### **다섯 모델의 취약 상품군**

- 각 모델의 추가 절대오차가 큰 상품군을 같은 기준선과 비교합니다.


In [ ]:
model_family_view = (all_family_scores.sort_values("excess_absolute_error", ascending=False)
        .groupby("model", sort=False).head(5)[
            ["model", "family", "actual_sales", "model_wape", "baseline_wape",
             "excess_absolute_error"]
        ])
model_family_view


### **다섯 모델의 취약 매장**

- 각 모델이 어떤 점포에서 기준선보다 불리한지 확인합니다.


In [ ]:
model_store_view = (all_store_scores.sort_values("excess_absolute_error", ascending=False)
        .groupby("model", sort=False).head(5)[
            ["model", "store_nbr", "actual_sales", "model_wape", "baseline_wape",
             "excess_absolute_error"]
        ])
model_store_view


In [ ]:
# 모델별 수요 규모의 개선폭을 한눈에 비교합니다.
# 색은 기준선 대비 WAPE 차이이며 판매량 비중은 위 표에서 함께 확인합니다.
heatmap = all_tier_scores.pivot(index="model", columns="demand_tier", values="gain_pp")
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heatmap, annot=True, fmt=".1f", cmap="RdYlGn", center=0, ax=ax)
ax.set(xlabel="7일 예상 수요 규모", ylabel="", title="4주 기준선 대비 7일 WAPE 개선 (pp)")
fig.tight_layout()
fig.savefig(analysis_dir / "all_models_demand_tier_gain.png", dpi=160)
plt.show()

### **모델별 낮은 수요 대체 효과**

각 모델에 대해 **그 모델의 Valid 낮은 수요**에서만 기준선이 더 정확한지 판단한 후, 선택한 규칙을 Test에 그대로 적용합니다. DL의 낮은 수요 오류를 고쳐도 전체 7일 WAPE가 기준선보다 높은지 확인하는 탐색적 비교입니다. Test는 이미 살펴본 구간이므로 여기서 선택한 모델이나 규칙의 독립적인 성능 증거로 사용하지 않습니다.

In [ ]:
all_rule_scores = model_fallback_scores(valid_model_weekly, test_model_weekly)
all_rule_scores.to_csv(analysis_dir / "all_models_weekly_fallback_exploration.csv", index=False)
fallback_view = (all_rule_scores.loc[all_rule_scores["phase"].eq("Test"), [
    "model", "method", "low_demand_fallback", "rows", "rmsle", "mae",
    "rmse", "wape", "bias",
]].sort_values(["model", "method"]))
fallback_view


## **7. 주간 검토 후보**

- LightGBM의 7일 예상 수요와 4주 기준선의 차이가 큰 매장×상품군을 표시합니다. 이는 **담당자가 검토할 후보**이지 자동 발주량이 아닙니다.
- 후보 선정에는 Test 정답을 쓰지 않습니다. `target_sales`는 과거 Test를 되돌아보는 표에만 표시하며, 미래 운영 시에는 아직 알 수 없습니다.

### **기준선보다 수요가 늘 것으로 본 사례**

- 마지막 Test 기준일 한 번에서 증가폭이 큰 검토 후보를 보여줍니다. 미래 운영 시 실제 판매량은 알 수 없습니다.


In [ ]:
weekly["forecast_change"] = weekly["prediction"] - weekly["sales_same_weekday_4w_mean"]
weekly["forecast_change_pct"] = 100 * weekly["forecast_change"] / weekly[
    "sales_same_weekday_4w_mean"
].replace(0, np.nan)

review_columns = [
    "date", "store_nbr", "family", "sales_same_weekday_4w_mean",
    "prediction", "forecast_change", "forecast_change_pct", "target_sales",
]
review_cycle = weekly.loc[weekly["date"].eq(weekly["date"].max())]
increasing_demand = review_cycle.nlargest(20, "forecast_change")[review_columns]
decreasing_demand = review_cycle.nsmallest(20, "forecast_change")[review_columns]

increasing_demand.to_csv(analysis_dir / "review_increasing_demand.csv", index=False)
decreasing_demand.to_csv(analysis_dir / "review_decreasing_demand.csv", index=False)
increasing_view = (increasing_demand.head(10))
increasing_view


### **기준선보다 수요가 줄 것으로 본 사례**

- 같은 기준일의 감소 후보입니다. 감소 예측도 실제 공급 축소 전 담당자 확인이 필요합니다.


In [ ]:
decreasing_view = (decreasing_demand.head(10))
decreasing_view


### **한 기준일의 사례 비교**

서로 겹치는 7일 예측을 여러 의사결정으로 세지 않도록 **Test의 마지막 기준일 한 번**만 사용합니다. 기준선 대비 변화가 큰 증가·감소 사례를 상품군별 한 건씩 골라 예측과 실제 7일 합계를 비교합니다. 사례 선택에는 실제 판매량을 쓰지 않습니다.

In [ ]:
one_cycle = weekly.loc[weekly["date"].eq(weekly["date"].max())].copy()
increasing_cases = one_cycle.sort_values("forecast_change", ascending=False).drop_duplicates("family").head(4)
decreasing_cases = one_cycle.sort_values("forecast_change").drop_duplicates("family").head(4)
cases = pd.concat([increasing_cases, decreasing_cases]).drop_duplicates("family")
cases["case"] = cases["store_nbr"].astype(str) + "호점 · " + cases["family"].astype(str)
cases[["date", "store_nbr", "family", "sales_same_weekday_4w_mean", "prediction",
       "target_sales", "forecast_change", "forecast_change_pct"]].to_csv(
    analysis_dir / "one_cycle_decision_cases.csv", index=False
)

case_plot = cases.melt(
    id_vars="case", value_vars=["sales_same_weekday_4w_mean", "prediction", "target_sales"],
    var_name="source", value_name="seven_day_sales",
)
case_plot["source"] = case_plot["source"].map({
    "sales_same_weekday_4w_mean": "4주 기준선", "prediction": "LightGBM", "target_sales": "실제",
})
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=case_plot, y="case", x="seven_day_sales", hue="source", ax=ax)
ax.set(xlabel="7일 합계 판매량", ylabel="", title=f"{one_cycle['date'].max():%Y-%m-%d} 기준일 검토 사례")
fig.tight_layout()
fig.savefig(analysis_dir / "one_cycle_decision_cases.png", dpi=160)
plt.show()

### **한 기준일의 사례 수치**

- 그림에 표시된 사례의 예측·기준선·과거 실제 수량을 수치로 확인합니다.


In [ ]:
case_view = (cases[["date", "store_nbr", "family", "forecast_change_pct", "target_sales"]])
case_view


## **8. 군집별 LightGBM 활용 범위**

- [02의 군집](02_demand_segments.ipynb)은 Valid 전 180일의 **다섯 판매 행동 특성**으로 정의합니다. 판촉 비율은 군집을 설명할 때만 사용합니다.
- `actual_share_pct`로 Test 판매 수량의 비중을 먼저 보고, LightGBM·기준선 WAPE와 `gain_pp`를 비교합니다. 양의 Bias는 과대예측입니다.
- 군집 번호는 재실행 시 달라질 수 있습니다. 특히 실제 판매량이 거의 없는 군집은 WAPE보다 **판매량 비중과 절대오차**를 우선 읽습니다.

In [ ]:
segment_path = REPO / "outputs" / "demand_segments" / "segment_features.parquet"
segment_result = None
if segment_path.is_file():
    segment_labels = pd.read_parquet(segment_path)
    assert "cluster_schema" in segment_labels, "새 군집 기준으로 02를 다시 실행하세요."
    assert segment_labels["cluster_schema"].eq("demand_5_without_promo_v1").all(), "02를 다시 실행하세요."
    assert pd.to_datetime(segment_labels["history_cutoff"]).eq("2017-07-18").all()
    assert not segment_labels.duplicated(["store_nbr", "family"]).any()
    weekly_segments = weekly.merge(
        segment_labels[["store_nbr", "family", "segment"]],
        on=["store_nbr", "family"], how="left", validate="many_to_one",
    )
    weekly_segments["segment"] = weekly_segments["segment"].fillna("이력 부족")
    segment_scores = group_scores(weekly_segments, ["segment"])
    segment_scores.to_csv(analysis_dir / "test_7day_error_by_segment.csv", index=False)
    segment_result = segment_scores[[
        "segment", "rows", "actual_share_pct", "lightgbm_wape", "baseline_wape",
        "gain_pp", "lightgbm_bias",
    ]].sort_values("segment")

segment_result

### **어느 군집이 중요하고, 어디서 모델이 개선되는가**

- 왼쪽은 Test의 실제 판매 수량 비중입니다. 오른쪽은 4주 기준선 대비 WAPE 개선폭이며, **0 아래는 모델이 더 나쁜 군집**입니다.
- 군집 번호를 고정된 이름처럼 해석하지 않습니다. 위 표의 수요 규모·0 판매 비율과 연결해 읽습니다.

In [ ]:
if segment_path.is_file():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.barplot(data=segment_scores, x="segment", y="actual_share_pct", ax=axes[0])
    sns.barplot(data=segment_scores, x="segment", y="gain_pp", ax=axes[1])
    axes[1].axhline(0, color="gray", linewidth=1)
    axes[0].set(title="Test 판매 수량 비중", xlabel="군집", ylabel="비중 (%)")
    axes[1].set(title="4주 기준선 대비 LightGBM 개선", xlabel="군집", ylabel="WAPE 개선 (pp)")
    fig.tight_layout()
    fig.savefig(analysis_dir / "test_segment_volume_and_gain.png", dpi=160)
    plt.show()

## **해석 원칙과 다음 검증**

- 전체 WAPE 개선과 **매장·상품군별 실패**를 함께 보고합니다. Bias가 양수이면 주간 총수요를 과대예측하는 방향입니다.
- 7일 합계 창이 서로 겹치므로 행 수만큼 독립적인 발주 사례를 검증한 것은 아닙니다. 마지막 기준일 시각화는 한 주기 사례입니다.
- 낮은 수요 대체 규칙은 Valid에서 정했지만 Test 결과를 이미 확인했습니다. [09 과거 시점 평가](09_lightgbm_historical_backtest.ipynb)에서 다른 날짜의 개선 반복 여부를 확인합니다.
- 다섯 모델의 수요 규모·상품군·매장별 결과는 같은 Test 행과 기준선으로 비교합니다. 모델별 입력 Feature가 달라 성능 차이를 구조만의 효과로 해석하지 않습니다.
- DL의 낮은 수요를 기준선으로 대체해도 높은 수요 구간이 기준선보다 나쁘면, 낮은 수요 오류만으로 전체 격차를 설명할 수 없습니다.
- 재고·공급·납기 자료가 없으므로 품절 감소, 비용 절감, 최적 발주량은 현재 결과로 검증할 수 없습니다.
- 군집별 Test 평가는 2017-07-18까지의 이력으로 만든 [02 수요 패턴 군집](02_demand_segments.ipynb)을 사용합니다. 군집은 학습 Feature가 아닙니다.
